# 01j — Extract Sentinel-2 Climatological Indices
**Data source:** [Sentinel-2 L2A COG](https://planetarycomputer.microsoft.com/dataset/sentinel-2-l2a) (Planetary Computer)

**Input:** `train_base.parquet`, `val_base.parquet` from notebook 00 output

**Output:** `sentinel.parquet` (one row per unique station)

### Temporal Constraint & Climatology Approach
Sentinel-2 was launched in late 2015. Our water quality sample dates span **2011 to 2015** (pre-Sentinel-2). 

To utilize the high-resolution (10m) bands of Sentinel-2, we use a **climatological baseline approach**:
- We extract the median spectral indices for each station over a fixed cloud-free window in **2020** (2020-01-01 to 2020-12-31).
- These act as **static spatial features** describing the local water body's baseline reflectance, vegetation index (EVI), water index (NDWI), and surface albedo.

**Estimated time:** ~15-20 min for ~170 stations (depends on Planetary Computer query times)

> Enable Internet in Kaggle settings.

In [ ]:
# Install required packages if running in Kaggle environment
!pip install -q geopandas pyarrow requests tqdm pystac-client planetary-computer odc-stac shapely

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, time, requests, logging
from tqdm import tqdm
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
import warnings
warnings.filterwarnings('ignore')

# === Logging Setup ===
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-5s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('01j_sentinel')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titleweight': 'bold', 'font.size': 11})

OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Column config
LAT_COL     = 'Latitude'
LON_COL     = 'Longitude'
STATION_COL = 'station_id'

# Helper function to find input file path in Kaggle
def find_file(filename, default_dir='/kaggle/working'):
    target = os.path.join(default_dir, filename)
    if os.path.exists(target):
        return target
    input_dir = '/kaggle/input'
    if os.path.exists(input_dir):
        for root, _, files in os.walk(input_dir):
            if filename in files:
                return os.path.join(root, filename)
    raise FileNotFoundError(f"File {filename} not found in input/working directories.")

In [ ]:
# Load base data to get unique stations
base_train_path = find_file('train_base.parquet')
base_val_path   = find_file('val_base.parquet')

train_base = pd.read_parquet(base_train_path)
val_base   = pd.read_parquet(base_val_path)
all_data   = pd.concat([train_base, val_base], ignore_index=True)

unique_stations = all_data.groupby(STATION_COL)[[LAT_COL, LON_COL]].first().reset_index()
log.info(f'Loaded unique stations: {len(unique_stations)}')

---
## Sentinel-2 STAC Extraction Logic

In [ ]:
def compute_sentinel_values(lat, lon, station_id, station_num, total_stations, retries=3):
    # ~100m buffer
    bbox_size = 0.00089831  
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2
    ]
    
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    
    # Search 2020 (climatology window)
    for attempt in range(retries):
        try:
            search = catalog.search(
                collections=["sentinel-2-l2a"],
                bbox=bbox,
                datetime="2020-01-01/2020-06-30",
                query={"eo:cloud_cover": {"lt": 10}},
            )
            items = search.item_collection()
            
            if not items:
                # Try second half of the year if first half has no low-cloud data
                search = catalog.search(
                    collections=["sentinel-2-l2a"],
                    bbox=bbox,
                    datetime="2020-07-01/2020-12-31",
                    query={"eo:cloud_cover": {"lt": 10}},
                )
                items = search.item_collection()
                
            if not items:
                raise ValueError("No low-cloud Sentinel-2 scenes found.")
            
            # Sort by cloud cover and pick the best scene
            selected_item = pc.sign(sorted(items, key=lambda x: x.properties.get("eo:cloud_cover", 100))[0])
            
            # Load bands (B02=blue, B03=green, B04=red, B08=nir, B11=swir16, B12=swir22)
            bands = ["B02", "B03", "B04", "B08", "B11", "B12"]
            data = stac_load([selected_item], bands=bands, bbox=bbox).isel(time=0)
            
            # Convert to reflectance float (scale factor = 10000)
            blue   = data["B02"].astype("float") / 10000.0
            green  = data["B03"].astype("float") / 10000.0
            red    = data["B04"].astype("float") / 10000.0
            nir    = data["B08"].astype("float") / 10000.0
            swir16 = data["B11"].astype("float") / 10000.0
            swir22 = data["B12"].astype("float") / 10000.0
            
            # Compute medians over buffer area
            m_blue   = float(blue.median(skipna=True).values)
            m_green  = float(green.median(skipna=True).values)
            m_red    = float(red.median(skipna=True).values)
            m_nir    = float(nir.median(skipna=True).values)
            m_swir16 = float(swir16.median(skipna=True).values)
            m_swir22 = float(swir22.median(skipna=True).values)
            
            # Compute Indices
            eps = 1e-10
            ndwi = (m_green - m_nir) / (m_green + m_nir + eps)
            evi = 2.5 * (m_nir - m_red) / (m_nir + 6.0 * m_red - 7.5 * m_blue + 1.0 + eps)
            # Liang (2001) Albedo proxy formula
            albedo = 0.356 * m_blue + 0.130 * m_red + 0.373 * m_nir + 0.085 * m_swir16 + 0.072 * m_swir22
            
            print(f'[{station_num}/{total_stations}] Station: {station_id[:15]} -> Success (NDWI={ndwi:.3f}, EVI={evi:.3f}, Albedo={albedo:.3f})')
            
            return {
                'sentinel_blue': m_blue, 'sentinel_green': m_green, 'sentinel_red': m_red,
                'sentinel_nir': m_nir, 'sentinel_swir16': m_swir16, 'sentinel_swir22': m_swir22,
                'sentinel_ndwi': ndwi, 'sentinel_evi': evi, 'sentinel_albedo': albedo
            }
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                print(f'[{station_num}/{total_stations}] Station: {station_id[:15]} -> FAILED (Error: {type(e).__name__})')
                
    return {
        'sentinel_blue': np.nan, 'sentinel_green': np.nan, 'sentinel_red': np.nan,
        'sentinel_nir': np.nan, 'sentinel_swir16': np.nan, 'sentinel_swir22': np.nan,
        'sentinel_ndwi': np.nan, 'sentinel_evi': np.nan, 'sentinel_albedo': np.nan
    }

---
## Perform Extraction

In [ ]:
sentinel_df = unique_stations.copy()
total = len(sentinel_df)
results = []

log.info(f'Starting Sentinel-2 climatology extraction for {total} stations...')
for idx, row in sentinel_df.iterrows():
    lat, lon = row[LAT_COL], row[LON_COL]
    station_id = row[STATION_COL]
    res = compute_sentinel_values(lat, lon, station_id, idx + 1, total)
    results.append(res)
    
sentinel_res_df = pd.DataFrame(results)
sentinel_df = pd.concat([sentinel_df.reset_index(drop=True), sentinel_res_df], axis=1)

display(sentinel_df.describe())

---
## Figure: Sentinel-2 NDWI Map

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
valid_mask = sentinel_df['sentinel_ndwi'].notna()

sc = ax.scatter(sentinel_df.loc[valid_mask, LON_COL], sentinel_df.loc[valid_mask, LAT_COL],
                c=sentinel_df.loc[valid_mask, 'sentinel_ndwi'], cmap='RdYlBu', s=70,
                edgecolors='gray', linewidths=0.3, alpha=0.9)
cbar = plt.colorbar(sc, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label('Sentinel-2 NDWI (Climatological)', fontsize=11)

# Mark failed/missing stations
if (~valid_mask).any():
    ax.scatter(sentinel_df.loc[~valid_mask, LON_COL], sentinel_df.loc[~valid_mask, LAT_COL],
              c='black', marker='x', s=100, linewidths=2, label='Failed/No Data', zorder=5)
    ax.legend(fontsize=10)

ax.set_xlim(16, 33); ax.set_ylim(-35, -22)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title(f'Sentinel-2 NDWI per Station\n{valid_mask.sum()}/{len(sentinel_df)} stations extracted')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_01j_sentinel_ndwi_map.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Save Output

In [ ]:
out_path = f'{OUTPUT_DIR}/sentinel.parquet'
sentinel_df.to_parquet(out_path, index=False)
size_kb = os.path.getsize(out_path) / 1024
log.info(f'Saved: {out_path} ({size_kb:.1f} KB, {len(sentinel_df)} rows)')
print('\n=== DONE ===')
print('Output: sentinel.parquet')
print('Next: add this notebook output as dataset input for 01e (Merge)')